## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

!export PATH="/home/<name>/.local/bin:$PATH"

# Create a virtual environment
!/home/<name>/.local/bin/uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install pandas prettyprint sympy numpy transformers accelerate vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [2]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "1"                    # CUDA_VISIBLE_DEVICES
PUBLIC_DATA_PATH   = "data/public.jsonl"
PRIVATE_DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional
import pandas as pd
import numpy as np

from pprint import pprint

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

W0430 23:29:22.416000 1745 torch/utils/cpp_extension.py:140] No CUDA runtime is found, using CUDA_HOME='/opt/conda'


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices - present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(PUBLIC_DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


In [5]:
from baseline.datasets import load_public_splits, load_private_set
from baseline.modeling import ModelConfig, load_transformers_model
from baseline.generation import GenerationConfig
from baseline.runner import run_problem_set

splits = load_public_splits(PUBLIC_DATA_PATH, val_frac=0.2, seed=414)

train_set = splits["train"]
val_set = splits["val"]
public_set = splits["public"]
private_set = load_private_set(PRIVATE_DATA_PATH)

pprint(train_set.summary())
pprint(val_set.summary())
pprint(private_set.summary())

{'n': 901,
 'n_answered': 901,
 'n_free_form': 601,
 'n_mcq': 300,
 'name': 'public_train'}
{'n': 225,
 'n_answered': 225,
 'n_free_form': 150,
 'n_mcq': 75,
 'name': 'public_val'}
{'n': 943, 'n_answered': 0, 'n_free_form': 643, 'n_mcq': 300, 'name': 'private'}


## 4. Modeling

In [6]:
model_config = ModelConfig(
    model_id="Qwen/Qwen3-4B-Thinking-2507",
    cache_dir=None,
    gpu_id=None,
    load_in_4bit=True,
    max_input_tokens=16384,
)

model_bundle = load_transformers_model(model_config)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
generation_config = GenerationConfig(
    max_new_tokens=1024,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
)

result = run_problem_set(
    problem_set=val_set.head(5),
    model_bundle=model_bundle,
    generation_config=generation_config,
    batch_size=1,
    score=True,
    output_jsonl_path="results/notebook_val5.jsonl",
)

result.summary

Generating:   0%|          | 0/5 [00:00<?, ?it/s]/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cpu/ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
df = pd.DataFrame(result.scored_rows)
df[["id", "is_mcq", "correct", "response"]].head()

In [ ]:
pd.DataFrame([result.timings | result.generations])